# Chapter Title Generator
**Problem Statement:** Authors often struggle with coming up with a creative, innovative, and/or fitting title for the chapters of their book or even the book itself. This process can interrupt writing flow and may be particularly difficult for new writers. 

**Proposed Methodology:** We propose a supervised deep learning approach using transformer-based sequence-to-sequence text generation models to solve this problem. We plan to train a model to generate a list of suitable titles


In [1]:
# Imports
import pandas as pd
from datasets import load_dataset, Dataset
import re
from transformers import BertTokenizerFast
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments, DataCollatorForSeq2Seq
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import gc
import torch

In [2]:
# Data Loading
stories_dataset = load_dataset("FareedKhan/1k_stories_100_genre", split="train")
cmu_dataset = load_dataset("textminr/cmu-book-summaries", split="train")
'''
if not os.path.exists("novel-chapter-dataset"):
    !git clone https://github.com/manestay/novel-chapter-dataset.git
else:
    print("Repository already cloned.")
'''
print(stories_dataset.column_names)
print(cmu_dataset.column_names)

['id', 'title', 'story', 'genre']
['title', 'author', 'pub_year', 'summary']


In [3]:
unified_data = []

for row in stories_dataset:
    unified_data.append({
        "input_text": row["story"], 
        "target_title": row["title"]
    })

for row in cmu_dataset:
    unified_data.append({
        "input_text": row["summary"], 
        "target_title": row["title"]
    })

df = pd.DataFrame(unified_data)

df = df.dropna().reset_index(drop=True)

In [4]:
tokenizer = BertTokenizerFast.from_pretrained("bert-base-uncased")

# Basic profanity filter using regex
def filter_profanity(text):
    bad_words_pattern = re.compile(r'\b(shit|fuck|bitch|cunt|slut|dick)\b', re.IGNORECASE)
    return not bool(bad_words_pattern.search(text))

# Instruction tuning formatter and chunker
def preprocess_function(examples):
    # Instruction tuning prompt
    inputs = ["Generate a chapter title for the following text: " + text for text in examples["input_text"]]
    
    # Keep inputs reasonably compact
    model_inputs = tokenizer(inputs, max_length=128, truncation=True, padding=False)
    
    # Titles should be short
    labels = tokenizer(text_target=examples["target_title"], max_length=15, truncation=True)
    
    cleaned_labels = []
    for label_set in labels["input_ids"]:
        cleaned_labels.append([(token if token != tokenizer.pad_token_id else -100) for token in label_set])
        
    model_inputs["labels"] = cleaned_labels
    return model_inputs

# Apply functions to dataset
clean_dataset = Dataset.from_pandas(df).train_test_split(test_size=0.1)
tokenized_datasets = clean_dataset.map(preprocess_function, batched=True)

# small_train_dataset = tokenized_datasets["train"].shuffle(seed=42).select(range(1,500))
# small_eval_dataset = tokenized_datasets["test"].shuffle(seed=42).select(range(200))

Map:   0%|          | 0/15803 [00:00<?, ? examples/s]

Map:   0%|          | 0/1756 [00:00<?, ? examples/s]

In [5]:
model_name = "google/flan-t5-small"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/308M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [9]:
# Force garbage collection and empty the MPS cache
gc.collect()
torch.mps.empty_cache()
print("MPS Memory Cache Cleared!")

data_collator = DataCollatorForSeq2Seq(
    tokenizer,
    model=model
)

training_args = Seq2SeqTrainingArguments(
    output_dir="./title_generator_model",
    eval_strategy="no",
    learning_rate=5e-5,
    per_device_eval_batch_size=8,       
    gradient_accumulation_steps=2,       
    weight_decay=0.01,
    save_total_limit=1,
    save_strategy="epoch",
    num_train_epochs=2,                
    predict_with_generate=False,
    fp16=False,            
    dataloader_pin_memory=False,   # Stops the Mac MPS warning and lag
    dataloader_num_workers=0,   
    logging_steps=10,          
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"], # Using the full ~15k dataset
    eval_dataset=tokenized_datasets["test"],   # Using the full ~1.7k test set
    data_collator=data_collator,
    processing_class=tokenizer,
)

# Launch the fine-tuning process
trainer.train()

MPS Memory Cache Cleared!


Step,Training Loss
10,22.616461
20,17.365759
30,15.768187
40,14.136191
50,13.899942
60,13.582431
70,13.295378
80,12.888707
90,13.291002
100,12.609485


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1976, training_loss=10.576557205756183, metrics={'train_runtime': 1490.9269, 'train_samples_per_second': 21.199, 'train_steps_per_second': 1.325, 'total_flos': 1468813822918656.0, 'train_loss': 10.576557205756183, 'epoch': 2.0})

In [ ]:
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
model.to(device)

def generate_creative_titles(story_text, num_options=3):
    prompt = "Generate a chapter title for the following text: " + story_text
    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    pad_id = tokenizer.pad_token_id
    eos_id = tokenizer.eos_token_id
    
    outputs = model.generate(
        inputs.input_ids,
        decoder_start_token_id=pad_id,
        pad_token_id=pad_id,
        eos_token_id=eos_id,
        
        max_length=10,     # Titles should be short
        
        do_sample=True,            
        temperature=0.8,           
        top_k=40,                    
        top_p=0.9,                  
        
        repetition_penalty=2.5,    
        no_repeat_ngram_size=1, # no repeating a word for a title
        
        num_return_sequences=num_options,
    )
    
    generated_titles = [tokenizer.decode(out, skip_special_tokens=True) for out in outputs]
    
    # Quick cleaning step: strip out loose commas/dots if the decoder gets lazy
    cleaned_titles = [title.strip(" .,;:-") for title in generated_titles]
    return cleaned_titles

In [32]:
# Cell: Testing with custom creative text

sample_story_1 = """
The atmospheric regulators hissed as Leo adjusted his oxygen mask. 
Outside the viewport, the crimson sands of the new colony were being swallowed by a massive dust storm. 
He checked the telemetry data one last time—if the shield generator failed tonight, the entire dome would depressurize.
"""

print("--- Generating Titles for Sample 1 ---")
suggested_titles_1 = generate_creative_titles(sample_story_1, num_options=3)

for i, title in enumerate(suggested_titles_1, 1):
    print(f"Option {i}: {title}")

--- Generating Titles for Sample 1 ---
Option 1: Leo Cabane
Option 2: The Clements II
Option 3: Leo II
